# 🏥 FloodShield AI — Model 6: Recommendation AI
**Logic:** District Risk Level (Model 5 output) + Real Hospital Stock → Resource Recommendation
**Recommends:** ORS, Medicine, Shelter, Emergency Kit, Medical Team deployment

### কীভাবে চালাবে:
1. Drive-এ `hospital_resources_csv.xls` থাকতে হবে
2. Model 5 notebook-এর `get_risk_score()` logic এখানে reuse করা হয়েছে (Model 2-এর artifact দিয়ে)
3. Runtime → Run all


## 1️⃣ Setup

In [ ]:
!pip install pandas joblib scikit-learn -q
from google.colab import drive
drive.mount('/content/drive')


## 2️⃣ Load Hospital Data + Model 2 (for risk score input)

In [ ]:
import pandas as pd
import numpy as np
import joblib

BASE = '/content/drive/MyDrive/FloodShieldAI'

hosp = pd.read_csv(f'{BASE}/data/hospital_resources_csv.xls')
outbreak_model = joblib.load(f'{BASE}/models/disease_outbreak_model.pkl')
outbreak_le = joblib.load(f'{BASE}/models/outbreak_risk_label_encoder.pkl')
outbreak_features = joblib.load(f'{BASE}/models/outbreak_model_features.pkl')

print("Hospital data:", hosp.shape)


## 3️⃣ Aggregate Hospital Stock (District-wise)

In [ ]:
hosp_district = hosp.groupby('district').agg({
    'beds':'sum','doctors':'sum','nurses':'sum',
    'ors_stock':'sum','iv_saline_stock':'sum','antibiotics_stock':'sum',
    'occupancy_percent':'mean'
}).reset_index()

ORS_LOW_THRESHOLD = hosp_district['ors_stock'].quantile(0.25)
IV_LOW_THRESHOLD = hosp_district['iv_saline_stock'].quantile(0.25)
ANTIBIOTIC_LOW_THRESHOLD = hosp_district['antibiotics_stock'].quantile(0.25)

print(f"ORS low threshold: {ORS_LOW_THRESHOLD:.0f}")
print(f"IV Saline low threshold: {IV_LOW_THRESHOLD:.0f}")
print(f"Antibiotics low threshold: {ANTIBIOTIC_LOW_THRESHOLD:.0f}")
hosp_district.head()


## 4️⃣ Recommendation Rule Engine

| Risk Level | Recommendation |
|---|---|
| Critical | ORS + IV Saline + Antibiotics + Emergency Medical Team + Shelter + Emergency Kits |
| High | ORS + IV Saline (stock check) + increased monitoring |
| Medium | Stock check + routine advisory |
| Low | Routine monitoring |


In [ ]:
def recommend_resources(district_name, risk_level):
    row = hosp_district[hosp_district['district']==district_name]
    if row.empty:
        return {"error": f"No hospital data found for {district_name}"}
    row = row.iloc[0]

    recommendations = []

    if risk_level == 'Critical':
        recommendations += ["Deploy Emergency Medical Team", "Prepare Emergency Shelter",
                             "Distribute Emergency Kits (household level)"]
        recommendations.append("URGENT: ORS restock needed" if row['ors_stock'] < ORS_LOW_THRESHOLD else "ORS stock adequate")
        recommendations.append("URGENT: IV Saline restock needed" if row['iv_saline_stock'] < IV_LOW_THRESHOLD else "IV Saline stock adequate")
        recommendations.append("URGENT: Antibiotics restock needed" if row['antibiotics_stock'] < ANTIBIOTIC_LOW_THRESHOLD else "Antibiotics stock adequate")

    elif risk_level == 'High':
        recommendations.append("Increase ORS supply" if row['ors_stock'] < ORS_LOW_THRESHOLD else "ORS stock adequate")
        recommendations.append("Increase IV Saline supply" if row['iv_saline_stock'] < IV_LOW_THRESHOLD else "IV Saline stock adequate")
        recommendations.append("Alert local health workers for increased monitoring")

    elif risk_level == 'Medium':
        recommendations.append("Monitor ORS stock" if row['ors_stock'] < ORS_LOW_THRESHOLD else "No immediate action needed")
        recommendations.append("Routine health advisory to community")

    else:
        recommendations.append("Routine monitoring, no special action needed")

    return {
        'district': district_name,
        'risk_level': risk_level,
        'hospital_beds_available': int(row['beds']),
        'occupancy_percent': round(row['occupancy_percent'],1),
        'recommendations': recommendations
    }

print(recommend_resources('Munshiganj', 'Critical'))


## 5️⃣ Combined Test — Model 5 output → Model 6 input
(Model 5-এর `get_risk_score()` এখানে ছোট করে reuse করছি শুধু test করার জন্য)


In [ ]:
def flood_severity_score(depth_cm):
    if depth_cm < 10: return 0
    elif depth_cm < 40: return 1
    elif depth_cm < 80: return 2
    else: return 3

def get_risk_score(rainfall_mm, temperature_c, humidity_percent, river_level_m, flood_depth_cm,
                    population_density, sanitation_score, water_quality_index,
                    previous_week_cases, previous_two_week_cases, season='Monsoon'):
    input_dict = {
        'rainfall_mm': rainfall_mm, 'temperature_c': temperature_c,
        'humidity_percent': humidity_percent, 'river_level_m': river_level_m,
        'flood_depth_cm': flood_depth_cm, 'population_density': population_density,
        'sanitation_score': sanitation_score, 'water_quality_index': water_quality_index,
        'previous_week_cases': previous_week_cases, 'previous_two_week_cases': previous_two_week_cases
    }
    season_cols = [c for c in outbreak_features if c.startswith('season_')]
    for col in season_cols:
        season_name = col.replace('season_', '')
        input_dict[col] = 1 if season_name == season else 0
    input_df = pd.DataFrame([input_dict])[outbreak_features]
    probs = outbreak_model.predict_proba(input_df)[0]
    outbreak_class = outbreak_le.classes_[probs.argmax()]
    outbreak_map = {'Medium':1, 'High':2, 'Critical':3}
    outbreak_score = outbreak_map.get(outbreak_class, 1)
    flood_score = flood_severity_score(flood_depth_cm)
    composite = 0.6*outbreak_score + 0.4*flood_score
    if composite < 1: level = 'Low'
    elif composite < 2: level = 'Medium'
    elif composite < 2.5: level = 'High'
    else: level = 'Critical'
    return {'final_risk_level': level, 'composite_score': round(composite,2)}

# Full test: Model 5 -> Model 6
risk = get_risk_score(rainfall_mm=220, temperature_c=29, humidity_percent=87, river_level_m=7.5,
                       flood_depth_cm=55, population_density=2100, sanitation_score=0.35,
                       water_quality_index=50, previous_week_cases=210, previous_two_week_cases=190,
                       season='Monsoon')
print("Model 5 output:", risk)

reco = recommend_resources('Munshiganj', risk['final_risk_level'])
print("Model 6 output:", reco)


## 6️⃣ Save

In [ ]:
import os
SAVE_DIR = f'{BASE}/models'
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(hosp_district, f'{SAVE_DIR}/hospital_district_aggregates.pkl')
joblib.dump({
    'ors': ORS_LOW_THRESHOLD, 'iv': IV_LOW_THRESHOLD, 'antibiotics': ANTIBIOTIC_LOW_THRESHOLD
}, f'{SAVE_DIR}/resource_thresholds.pkl')

print(f"✅ Saved to: {SAVE_DIR}")


---
## ✅ পরের ধাপ
`recommend_resources()` ফাংশন backend-এর `/recommend` endpoint হবে, যেটা Model 5-এর output input হিসেবে নেবে।

**সৎ Note:** এটাও rule-based decision engine — ML training দরকার নাই, কারণ recommendation logic explainable/auditable রাখা এই ধরনের critical health system-এ গুরুত্বপূর্ণ।
